# Healthcare Claims Analysis
## January – April 2026

This notebook provides a professional analysis of medical claims data, focusing on:
- Service type and benefit breakdowns
- Provider intelligence (hospital spending patterns)
- Monthly trends
- Patient switching behaviour and hospital retention

All analysis is based on the `VISIT_KEY` concept: one member + one arrival date + one service type = one unique visit.

## 1. Setup and Data Loading

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

# Load data
df = pd.read_csv('../data/visits for an-april-2026.csv')

# Standardise column names (strip spaces)
df.columns = df.columns.str.strip()

# Convert date columns to datetime
df['ARRIVAL DATE'] = pd.to_datetime(df['ARRIVAL DATE'])
df['TRANSACTION DATE'] = pd.to_datetime(df['TRANSACTION DATE'])

# Convert amount to numeric (coerce errors)
df['AMOUNT'] = pd.to_numeric(df['AMOUNT'], errors='coerce')

print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Dataset shape: 22945 rows, 45 columns


,EDI_NO,CLAIM ID,CENTRAL ID,CLAIM TYPE,SCHEME,MEMBER NUMBER,INTEG MEMBER NUMBER,OTHER NUMBER,OFFICE BRANCH,CARD SERIAL,...,GLOBAL INVOICE NUMBER,PRINCIPAL NAMES,PRINCIPAL MEMBER NUMBER,PRINCIPAL OTHER NUMBER,RELATIONSHIP TO PRINCIPAL,PHONE NUMBER,PHONE NUMBER2,PARENT POOL,PARENT POOL DESCRIPTION,STATUS
0,31076359.0,469068207.0,386934712.0,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-103969-00,2022899.0,103969,NaN,VCKE0001602153,...,NaN,NaN,NaN,NaN,NaN,2.547252e+11,NaN,NaN,NaN,NaN
1,31349211.0,469370325.0,387383730.0,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-108707-00,2016618.0,108707,NaN,VCKE0001603519,...,EQA-04300-68112,NaN,NaN,NaN,NaN,2.547261e+11,NaN,NaN,NaN,NaN
2,32314856.0,470441199.0,401302495.0,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-57710-00,2015303.0,57710,NaN,VCKE0001612065,...,EQA-06520-5730,NaN,NaN,NaN,NaN,2.547225e+11,NaN,NaN,NaN,NaN
3,32060906.0,470158051.0,390685385.0,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-64773-04,2020294.0,64773,NaN,VCKE0001617322,...,EQA-02500-83921,NaN,NaN,NaN,NaN,2.547218e+11,NaN,NaN,NaN,NaN
4,31028192.0,469007823.0,386855023.0,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-67446-01,2408256.0,67446,NaN,VCKE0001617946,...,0,NaN,NaN,NaN,NaN,2.547113e+11,NaN,NaN,NaN,NaN


## 2. Create Visit Key

A visit is defined as a unique combination of member number, arrival date, and service type.

In [3]:
df['VISIT_KEY'] = (
    df['MEMBER NUMBER'].astype(str) + '_' +
    df['ARRIVAL DATE'].dt.strftime('%Y-%m-%d') + '_' +
    df['SERVICE TYPE'].astype(str)
)

print("VISIT_KEY created. Example:", df['VISIT_KEY'].iloc[0])

VISIT_KEY created. Example: DEFMIS-103969-00_2026-01-07_OP


## 3. High‑Level Summary by Service Type

In [4]:
# Aggregate by SERVICE TYPE
service_summary = df.groupby('SERVICE TYPE').agg(
    TOTAL_AMOUNT=('AMOUNT', 'sum'),
    UNIQUE_VISITS=('VISIT_KEY', 'nunique')
).reset_index()

# Add percentage and average cost per visit
grand_total = service_summary['TOTAL_AMOUNT'].sum()
service_summary['SHARE (%)'] = (service_summary['TOTAL_AMOUNT'] / grand_total * 100).round(2)
service_summary['AVG_COST_PER_VISIT'] = (service_summary['TOTAL_AMOUNT'] / service_summary['UNIQUE_VISITS']).round(2)

# Format for display
display_summary = service_summary.copy()
display_summary['TOTAL_AMOUNT'] = display_summary['TOTAL_AMOUNT'].apply(lambda x: f"Ksh {x:,.0f}")
display_summary['AVG_COST_PER_VISIT'] = display_summary['AVG_COST_PER_VISIT'].apply(lambda x: f"Ksh {x:,.2f}")
display_summary['UNIQUE_VISITS'] = display_summary['UNIQUE_VISITS'].apply(lambda x: f"{x:,}")
display_summary['SHARE (%)'] = display_summary['SHARE (%)'].astype(str) + '%'

print("### Service Type Summary\n")
display(display_summary)

# Bar chart
fig = px.bar(
    service_summary,
    x='SERVICE TYPE',
    y='TOTAL_AMOUNT',
    title='Total Amount by Service Type',
    text=service_summary['TOTAL_AMOUNT'].apply(lambda x: f'{x/1e6:.1f}M'),
    color='TOTAL_AMOUNT',
    color_continuous_scale='Viridis'
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_tickformat=',.0f', height=450)
fig.show()

### Service Type Summary



,SERVICE TYPE,TOTAL_AMOUNT,UNIQUE_VISITS,SHARE (%),AVG_COST_PER_VISIT
0,IP,"Ksh 183,861,656","1,483",56.84%,"Ksh 123,979.54"
1,OP,"Ksh 139,592,634","18,331",43.16%,"Ksh 7,615.11"


## 4. Breakdown by Benefit Description

In [5]:
benefit_summary = df.groupby('BENEFIT DESC').agg(
    UNIQUE_VISITS=('VISIT_KEY', 'nunique'),
    TOTAL_AMOUNT=('AMOUNT', 'sum')
).reset_index()

benefit_summary['AVG_COST_PER_VISIT'] = (benefit_summary['TOTAL_AMOUNT'] / benefit_summary['UNIQUE_VISITS']).round(2)
benefit_summary = benefit_summary.sort_values('TOTAL_AMOUNT', ascending=False)

# Format
benefit_summary['TOTAL_AMOUNT'] = benefit_summary['TOTAL_AMOUNT'].apply(lambda x: f"Ksh {x:,.2f}")
benefit_summary['AVG_COST_PER_VISIT'] = benefit_summary['AVG_COST_PER_VISIT'].apply(lambda x: f"Ksh {x:,.2f}")

print("### Benefit Description Summary\n")
display(benefit_summary)

### Benefit Description Summary



,BENEFIT DESC,UNIQUE_VISITS,TOTAL_AMOUNT,AVG_COST_PER_VISIT
0,IN PATIENT OVERALL / HOSPITALIZATION/ACCOMODATION,1483,"Ksh 183,861,656.08","Ksh 123,979.54"
3,OUT PATIENT OVERALL,17502,"Ksh 129,984,938.60","Ksh 7,426.86"
2,OUT PATIENT OPTICAL,629,"Ksh 6,516,887.97","Ksh 10,360.71"
1,OUT PATIENT DENTAL,425,"Ksh 3,090,807.71","Ksh 7,272.49"


## 5. Provider Intelligence (Hospital × Benefit Matrix)

Shows total amount spent per hospital per benefit category.

In [6]:
# Pivot table: Hospital x Benefit Desc
provider_benefit = pd.pivot_table(
    df,
    index='MAIN HOSPITAL',
    columns='BENEFIT DESC',
    values='AMOUNT',
    aggfunc='sum',
    fill_value=0
)

# Add total column and sort
provider_benefit['TOTAL_COST'] = provider_benefit.sum(axis=1)
provider_benefit = provider_benefit.sort_values('TOTAL_COST', ascending=False)

# Format total cost for display
provider_benefit_display = provider_benefit.copy()
provider_benefit_display['TOTAL_COST'] = provider_benefit_display['TOTAL_COST'].apply(lambda x: f"Ksh {x:,.2f}")

print("### Top 10 Hospitals by Total Spend\n")
display(provider_benefit_display.head(10))

### Top 10 Hospitals by Total Spend



BENEFIT DESC,IN PATIENT OVERALL / HOSPITALIZATION/ACCOMODATION,OUT PATIENT DENTAL,OUT PATIENT OPTICAL,OUT PATIENT OVERALL,TOTAL_COST
MAIN HOSPITAL,,,,,
ULINZI PRIME HEALTH SERVICES FUND (UPHSF),22273209.58,754944.53,32889.81,3.280456e+07,"Ksh 55,865,601.89"
NAIROBI HOSP REFERRAL,13925819.75,62489.00,0.00,6.485377e+06,"Ksh 20,473,685.33"
NAIROBI WEST HOSP,15165525.46,75789.00,0.00,3.660974e+06,"Ksh 18,902,288.96"
THE KAREN HOSP REFERRAL,10892671.00,42932.00,21000.00,3.934186e+06,"Ksh 14,890,788.60"
ST LUKE ORTHOPEADIC ELD,11773234.32,38841.00,32566.00,2.722963e+06,"Ksh 14,567,604.36"
MONALIFE PHARMACEUTICALS LTD,0.00,0.00,0.00,1.443821e+07,"Ksh 14,438,207.60"
THE AGA KHAN HOSP KIS,9151723.87,48970.00,43575.00,3.995934e+06,"Ksh 13,240,202.68"
NAIROBI SOUTH HOSPITAL,10854594.55,29250.00,0.00,5.679648e+05,"Ksh 11,451,809.38"
AGA KHAN HOSP MOMBASA,7461141.98,61725.00,6825.00,1.451296e+06,"Ksh 8,980,987.66"


## 6. Monthly Trends

In [7]:
# Create month-period column
df['YEAR_MONTH'] = df['TRANSACTION DATE'].dt.to_period('M')

# Aggregate monthly
monthly = df.groupby('YEAR_MONTH').agg(
    TOTAL_AMOUNT=('AMOUNT', 'sum'),
    TOTAL_CLAIMS=('CLAIM ID', 'count'),
    UNIQUE_VISITS=('VISIT_KEY', 'nunique')
).reset_index()

monthly['YEAR_MONTH_STR'] = monthly['YEAR_MONTH'].astype(str)
monthly = monthly.sort_values('YEAR_MONTH')

print("### Monthly Summary\n")
display(monthly[['YEAR_MONTH_STR', 'TOTAL_AMOUNT', 'TOTAL_CLAIMS', 'UNIQUE_VISITS']])

# Line chart (dual axis)
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=monthly['YEAR_MONTH_STR'], y=monthly['TOTAL_AMOUNT'],
                         name='Total Amount (Ksh)', mode='lines+markers'),
              secondary_y=False)
fig.add_trace(go.Scatter(x=monthly['YEAR_MONTH_STR'], y=monthly['UNIQUE_VISITS'],
                         name='Unique Visits', mode='lines+markers'),
              secondary_y=True)

fig.update_layout(title='Monthly Trends: Amount vs Unique Visits', height=450)
fig.update_yaxes(title_text='Amount (Ksh)', secondary_y=False)
fig.update_yaxes(title_text='Number of Visits', secondary_y=True)
fig.show()

### Monthly Summary



,YEAR_MONTH_STR,TOTAL_AMOUNT,TOTAL_CLAIMS,UNIQUE_VISITS
0,2026-01,7.440340e+07,5936,5083
1,2026-02,8.751369e+07,5593,4835
2,2026-03,8.684285e+07,5755,4981
3,2026-04,7.469435e+07,5660,4936


## 7. Patient Switching Analysis

We analyse patients who **start** at one hospital and later visit a **different** hospital. The analysis identifies:
- Hospitals that lose the most patients (worst retention)
- Where those patients go (flow analysis)
- Retention rates based on first vs last hospital

In [8]:
# Helper functions to get hospital at min/max date
def get_first_hospital(group):
    return group.loc[group['ARRIVAL DATE'].idxmin(), 'MAIN HOSPITAL']

def get_last_hospital(group):
    return group.loc[group['ARRIVAL DATE'].idxmax(), 'MAIN HOSPITAL']

# Member-level summary
member_first = df.groupby('MEMBER NUMBER').apply(get_first_hospital).reset_index(name='FIRST_HOSPITAL')
member_last = df.groupby('MEMBER NUMBER').apply(get_last_hospital).reset_index(name='LAST_HOSPITAL')
member_visits = df.groupby('MEMBER NUMBER')['CLAIM ID'].count().reset_index(name='VISIT_COUNT')
member_dates = df.groupby('MEMBER NUMBER')['ARRIVAL DATE'].agg(['min', 'max']).reset_index()
member_dates.columns = ['MEMBER NUMBER', 'FIRST_DATE', 'LAST_DATE']

# Merge
member_summary = member_first.merge(member_last, on='MEMBER NUMBER')
member_summary = member_summary.merge(member_visits, on='MEMBER NUMBER')
member_summary = member_summary.merge(member_dates, on='MEMBER NUMBER')
member_summary['SWITCHED'] = member_summary['FIRST_HOSPITAL'] != member_summary['LAST_HOSPITAL']

# Aggregate by first hospital
hospital_switch = member_summary.groupby('FIRST_HOSPITAL').agg(
    PATIENTS_STARTED=('MEMBER NUMBER', 'nunique'),
    PATIENTS_LOST=('SWITCHED', 'sum'),
    TOTAL_VISITS=('VISIT_COUNT', 'sum')
).reset_index()

hospital_switch['RETENTION_RATE'] = (1 - hospital_switch['PATIENTS_LOST'] / hospital_switch['PATIENTS_STARTED']) * 100
hospital_switch = hospital_switch.sort_values('PATIENTS_LOST', ascending=False)

print("### Worst Hospitals (Most Patients Lost)\n")
worst_display = hospital_switch.head(10)[['FIRST_HOSPITAL', 'PATIENTS_STARTED', 'PATIENTS_LOST', 'RETENTION_RATE']]
worst_display.columns = ['Hospital', 'Patients Started', 'Patients Lost', 'Retention Rate (%)']
worst_display['Retention Rate (%)'] = worst_display['Retention Rate (%)'].round(1)
display(worst_display)

print("### Best Hospitals (Most Patients Retained)\n")
best_display = hospital_switch.sort_values('PATIENTS_STARTED', ascending=False).head(10)
best_display = best_display[['FIRST_HOSPITAL', 'PATIENTS_STARTED', 'PATIENTS_LOST', 'RETENTION_RATE']]
best_display.columns = ['Hospital', 'Patients Started', 'Patients Lost', 'Retention Rate (%)']
best_display['Retention Rate (%)'] = best_display['Retention Rate (%)'].round(1)
display(best_display)

# Bar chart: worst by patients lost
fig_worst = px.bar(
    hospital_switch.head(10),
    x='PATIENTS_LOST',
    y='FIRST_HOSPITAL',
    orientation='h',
    title='Hospitals with Highest Patient Loss (first ≠ last hospital)',
    text='PATIENTS_LOST',
    color='PATIENTS_LOST',
    color_continuous_scale='Reds'
)
fig_worst.update_traces(texttemplate='%{text}', textposition='outside')
fig_worst.update_layout(height=500, margin=dict(l=150))
fig_worst.show()

### Worst Hospitals (Most Patients Lost)



,Hospital,Patients Started,Patients Lost,Retention Rate (%)
125,ULINZI PRIME HEALTH SERVICES FUND (UPHSF),1648,217,86.8
26,EQUITY AFIA RONGAI,663,140,78.9
62,NAIROBI WEST HOSP,152,46,69.7
0,AAR HEALTHCARE NAIROBI,197,41,79.2
60,NAIROBI HOSP REFERRAL,148,34,77.0
117,THE KAREN HOSP REFERRAL,114,30,73.7
55,MONALIFE PHARMACEUTICALS LTD,91,27,70.3
100,ST LUKE ORTHOPEADIC ELD,124,23,81.5
42,LIONS SIGHT FIRST EYE,60,22,63.3
114,THE AGA KHAN HOSP KIS,158,21,86.7


### Best Hospitals (Most Patients Retained)



,Hospital,Patients Started,Patients Lost,Retention Rate (%)
125,ULINZI PRIME HEALTH SERVICES FUND (UPHSF),1648,217,86.8
26,EQUITY AFIA RONGAI,663,140,78.9
0,AAR HEALTHCARE NAIROBI,197,41,79.2
114,THE AGA KHAN HOSP KIS,158,21,86.7
62,NAIROBI WEST HOSP,152,46,69.7
60,NAIROBI HOSP REFERRAL,148,34,77.0
100,ST LUKE ORTHOPEADIC ELD,124,23,81.5
96,ST FRANCIS COMMUNITY KASARANI,122,21,82.8
117,THE KAREN HOSP REFERRAL,114,30,73.7
8,BISHOP KIOKO CATHOLIC,111,15,86.5


### Where Do Lost Patients Go? (Sankey Diagram)

For the top 5 hospitals with the most patient losses, we visualise the flow to their new hospitals.

In [9]:
top_origins = hospital_switch.head(5)['FIRST_HOSPITAL'].tolist()
flow_data = member_summary[member_summary['FIRST_HOSPITAL'].isin(top_origins) & member_summary['SWITCHED']]
flow_sum = flow_data.groupby(['FIRST_HOSPITAL', 'LAST_HOSPITAL']).size().reset_index(name='count')
flow_sum = flow_sum.sort_values('count', ascending=False).head(15)

if not flow_sum.empty:
    labels = list(set(flow_sum['FIRST_HOSPITAL'].unique()) | set(flow_sum['LAST_HOSPITAL'].unique()))
    label_to_index = {label: i for i, label in enumerate(labels)}
    source = [label_to_index[h] for h in flow_sum['FIRST_HOSPITAL']]
    target = [label_to_index[h] for h in flow_sum['LAST_HOSPITAL']]
    value = flow_sum['count']

    fig_sankey = go.Figure(data=[go.Sankey(
        node=dict(pad=15, thickness=20, line=dict(color='black', width=0.5), label=labels),
        link=dict(source=source, target=target, value=value)
    )])
    fig_sankey.update_layout(title='Patient Flow: First Hospital → Last Hospital (Top 5 Origins)', height=600)
    fig_sankey.show()
else:
    print("Not enough flow data for Sankey diagram.")

### 7‑Day Switching (Rapid Defection)

We identify patients who switch to a different hospital within **7 days** of a previous visit. This can indicate dissatisfaction or urgent transfers.

In [10]:
# Sort by member and date
df_sorted = df.sort_values(['MEMBER NUMBER', 'ARRIVAL DATE'])

switches = []
for member, group in df_sorted.groupby('MEMBER NUMBER'):
    if len(group) < 2:
        continue
    prev_row = None
    for idx, row in group.iterrows():
        if prev_row is not None:
            if row['MAIN HOSPITAL'] != prev_row['MAIN HOSPITAL']:
                date_diff = (row['ARRIVAL DATE'] - prev_row['ARRIVAL DATE']).days
                if 0 < date_diff <= 7:
                    switches.append({
                        'MEMBER NUMBER': member,
                        'HOSPITAL_FROM': prev_row['MAIN HOSPITAL'],
                        'HOSPITAL_TO': row['MAIN HOSPITAL'],
                        'DAYS_BETWEEN': date_diff
                    })
        prev_row = row

switches_df = pd.DataFrame(switches)

if not switches_df.empty:
    # Count patients who switched from each hospital
    switch_counts = switches_df.groupby('HOSPITAL_FROM')['MEMBER NUMBER'].nunique().reset_index()
    switch_counts.columns = ['HOSPITAL', 'PATIENTS_LEFT_7D']
    
    # Total patients per hospital
    total_patients = df.groupby('MAIN HOSPITAL')['MEMBER NUMBER'].nunique().reset_index()
    total_patients.columns = ['HOSPITAL', 'TOTAL_PATIENTS']
    
    # Merge and compute rate
    switch_7d = total_patients.merge(switch_counts, on='HOSPITAL', how='left').fillna(0)
    switch_7d['SWITCH_RATE_7D'] = (switch_7d['PATIENTS_LEFT_7D'] / switch_7d['TOTAL_PATIENTS']) * 100
    switch_7d = switch_7d.sort_values('SWITCH_RATE_7D', ascending=False)
    
    print("### Hospitals with Highest 7‑Day Patient Defection\n")
    display(switch_7d.head(10)[['HOSPITAL', 'TOTAL_PATIENTS', 'PATIENTS_LEFT_7D', 'SWITCH_RATE_7D']].round(1))
else:
    print("No 7‑day switching events found.")

### Hospitals with Highest 7‑Day Patient Defection



,HOSPITAL,TOTAL_PATIENTS,PATIENTS_LEFT_7D,SWITCH_RATE_7D
7,BESTCARE HOSPITAL LIMITED,3,2.0,66.7
94,SPINE CLINIC AFRICA LTD,5,2.0,40.0
15,CHIROMO LANE MEDICAL CENTRE,8,3.0,37.5
111,STARKEY HEARING TECHNOLOGIES LTD,8,3.0,37.5
39,KILOME MATERNITY NURSING HOME,6,2.0,33.3
122,THE MITUNGUU HOSP LTD,3,1.0,33.3
24,ELGON VIEW HOSP ELDORET,9,3.0,33.3
13,CANCER CARE CENTRE,15,5.0,33.3
37,KENYATTA NATIONAL HOSP,15,4.0,26.7
38,KENYATTA UNIVERSITY HOSPITAL (KUTRR),23,6.0,26.1


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22945 entries, 0 to 22944
Data columns (total 47 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   EDI_NO                     22787 non-null  float64       
 1   CLAIM ID                   22944 non-null  float64       
 2   CENTRAL ID                 22944 non-null  float64       
 3   CLAIM TYPE                 22944 non-null  object        
 4   SCHEME                     22944 non-null  object        
 5   MEMBER NUMBER              22944 non-null  object        
 6   INTEG MEMBER NUMBER        22943 non-null  float64       
 7   OTHER NUMBER               22944 non-null  object        
 8   OFFICE BRANCH              39 non-null     object        
 9   CARD SERIAL                22754 non-null  object        
 10  PATIENT NAME               22944 non-null  object        
 11  DOB                        22944 non-null  object        
 12  CAT 

In [12]:
# Identify members with both OP and IP on the same day

# Group by member number and arrival date
same_day = df.groupby(['MEMBER NUMBER', 'ARRIVAL DATE']).filter(
    lambda g: set(g['SERVICE TYPE']) == {'OP', 'IP'}
)

# Show result
print(f"Found {same_day['MEMBER NUMBER'].nunique()} members with both OP and IP on the same day.")
print(f"Total rows (claims) involved: {len(same_day)}")

# For each such day, sort by transaction time to see order (OP → IP or IP → OP)
same_day_sorted = same_day.sort_values(['MEMBER NUMBER', 'ARRIVAL DATE', 'TRANSACTION DATE'])

# Display first few records
same_day_sorted[['MEMBER NUMBER', 'ARRIVAL DATE', 'SERVICE TYPE', 'TRANSACTION DATE', 'AMOUNT']].head(10)

Found 124 members with both OP and IP on the same day.
Total rows (claims) involved: 424


,MEMBER NUMBER,ARRIVAL DATE,SERVICE TYPE,TRANSACTION DATE,AMOUNT
18400,DEFMIS-10031-01,2026-03-12,IP,2026-03-12,15125.29
18401,DEFMIS-10031-01,2026-03-12,OP,2026-03-12,6460.00
17715,DEFMIS-10031-01,2026-03-18,OP,2026-03-18,7254.00
17822,DEFMIS-10031-01,2026-03-18,IP,2026-03-18,5400.00
7659,DEFMIS-10031-01,2026-03-24,OP,2026-03-24,8740.00
17175,DEFMIS-10031-01,2026-03-24,IP,2026-03-24,5400.00
15729,DEFMIS-10031-01,2026-04-01,OP,2026-04-01,7501.00
15906,DEFMIS-10031-01,2026-04-01,IP,2026-04-01,5400.00
14332,DEFMIS-10031-01,2026-04-09,OP,2026-04-09,6460.00
14539,DEFMIS-10031-01,2026-04-09,IP,2026-04-09,5400.00


In [13]:
# Identify same-day OP-IP events per member-date
same_day = df.groupby(['MEMBER NUMBER', 'ARRIVAL DATE']).filter(
    lambda g: set(g['SERVICE TYPE']) == {'OP', 'IP'}
)

# For each event, we need to assign a MAIN HOSPITAL. 
# Since the question asks for "MAIN HOSPITAL with these cases", 
# we'll assume the hospital is the one where the IP claim occurred (or we can use the first hospital of the day).
# Let's use the hospital from the IP claim (if there are multiple IP claims, take the first).
ip_records = same_day[same_day['SERVICE TYPE'] == 'IP'].drop_duplicates(subset=['MEMBER NUMBER', 'ARRIVAL DATE'])
event_hospital = ip_records[['MEMBER NUMBER', 'ARRIVAL DATE', 'MAIN HOSPITAL']]

# Count events per hospital
hospital_counts = event_hospital.groupby('MAIN HOSPITAL').size().reset_index(name='same_day_OP_IP_events')
hospital_counts = hospital_counts.sort_values('same_day_OP_IP_events', ascending=False)

# Top 10
top10 = hospital_counts.head(10)
print("Top 10 Main Hospitals with Same-Day OP → IP (or both) Cases")
print(top10.to_string(index=False))

# Optional: also show the total number of unique members involved per hospital
member_counts = same_day.groupby('MAIN HOSPITAL')['MEMBER NUMBER'].nunique().reset_index(name='unique_members')
top10_members = member_counts.sort_values('unique_members', ascending=False).head(10)
print("\nTop 10 Hospitals by Unique Members with Same-Day OP-IP")
print(top10_members.to_string(index=False))

Top 10 Main Hospitals with Same-Day OP → IP (or both) Cases
                            MAIN HOSPITAL  same_day_OP_IP_events
                        NAIROBI WEST HOSP                     29
                    LIONS SIGHT FIRST EYE                     23
ULINZI PRIME HEALTH SERVICES FUND (UPHSF)                     17
                 DR AGARWALS EYE HOSPITAL                     16
           BRISTOL PARK HEALTHCARE CENTRE                     14
                    NAIROBI HOSP REFERRAL                     11
                  THE KAREN HOSP REFERRAL                      8
                  ST LUKE ORTHOPEADIC ELD                      8
                         SABATIA EYE HOSP                      6
                    BISHOP KIOKO CATHOLIC                      5

Top 10 Hospitals by Unique Members with Same-Day OP-IP
                            MAIN HOSPITAL  unique_members
                    LIONS SIGHT FIRST EYE              18
ULINZI PRIME HEALTH SERVICES FUND (UPHSF)            

## 8. Summary & Recommendations

- **Service Type**: IP visits account for ~57% of total spend despite being only ~7% of visits. IP average cost (~124k) is much higher than OP (~7.6k).
- **Top Providers**: A small number of hospitals (UPHSF, Nairobi Hospital, Nairobi West) dominate total spending.
- **Patient Retention**: Several large hospitals lose hundreds of patients over the period. Sankey diagrams show where these patients go.
- **Rapid Switching**: Some hospitals have 7‑day defection rates >20%, suggesting potential quality or operational issues.

**Next Steps**:
- Investigate root causes for high-switch hospitals (e.g., patient complaints, wait times, service gaps).
- Negotiate with high‑retention hospitals for preferred partnerships.
- Monitor monthly trends to detect sudden changes in provider market share.